In [1]:
# --- output directories (created relative to repo root) ---
from pathlib import Path as _P
for _d in ('figures', 'figures/decoupling'):
    _P(_d).mkdir(parents=True, exist_ok=True)

# nanoPhos - SPEC / enrichment decoupling experiment

**Reviewer 1 novelty rebuttal.** Isolates the specific contribution of the nanoPhos
phosphoenrichment relative to the SPEC protein preparation, using a factor-decoupling
design (protein prep x phosphoenrichment):

| Branch | Prep | Enrichment |
|---|---|---|
| **classic** | SPEC | nanoPhos-optimized (Fe(III)-NTA; 200 mM NaCl / 10% ACN load; direct Evotip elution) |
| **hybrid** | SPEC | Agilent-standard AssayMAP Fe(III)-NTA |
| **wCleanup** | in-solution + SDB-RPS + SpeedVac | Agilent-standard |
| **woCleanup** | in-solution, crude lysate, no cleanup | Agilent-standard |

Pairwise contrasts each isolate one factor: **classic-hybrid** = enrichment contribution
(SPEC held constant, the direct R1 answer); **hybrid-wCleanup** = SPEC-prep contribution;
**wCleanup-woCleanup** = cleanup effect; **classic-wCleanup** = full nanoPhos vs a standard
conventional workflow (Reviewer 1's suggested "in-solution + platform" comparison).

Depth metrics per Bekker-Jensen 2020: unique **Class I** localized phosphosites
(loc prob >= 0.75) and **phosphopeptide precursors**, per run (n = 4) across 10-1000 ng
(4 branches x 7 inputs x 4 reps = 112 runs).

### Reproducibility
All heavy preprocessing lives in `src/decoupling_preprocess.py`. This notebook reads the
committed result tables in `data/decoupling_*.csv` by default, so every panel reproduces
instantly. To rebuild the tables from the raw Spectronaut reports, set `REGENERATE = True`
below; the raw precursor + Class I reports (~2.7 GB) are read from the relative path
`pride_data/analysis_data/revision/decoupling/` (extract the corresponding MassIVE archive
there).

In [2]:
import sys, os, importlib
sys.path.insert(0, os.path.abspath('src'))
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from scipy import stats
from core import _hex_to_rgba
import decoupling_preprocess as dp
importlib.reload(dp)

<module 'decoupling_preprocess' from 'd:\\Projects\\nanoPhos_env\\src\\decoupling_preprocess.py'>

In [3]:
# --- configuration -------------------------------------------------------
REGENERATE  = False                                          # True -> rebuild data/ tables from raw reports
RAW_DIR     = 'pride_data/analysis_data/revision/decoupling' # raw Spectronaut reports (MassIVE); relative
RESULTS_DIR = 'data'                                         # committed result CSVs (data/decoupling_*.csv)

BRANCH_ORDER = dp.BRANCH_ORDER          # ['classic', 'hybrid', 'wCleanup', 'woCleanup']
INPUT_ORDER  = dp.INPUT_ORDER           # [10, 20, 50, 100, 200, 500, 1000]

# classic = brand deep red (hero); the three conventional branches in cooler tones
BRANCH_COLORS = {
    'classic':   '#8A0000',   # nanoPhos (SPEC + optimized enrichment)
    'hybrid':    '#2C6FB0',   # SPEC + standard Agilent enrichment  (Reviewer 1's premise)
    'wCleanup':  '#E1812C',   # in-solution + cleanup + standard
    'woCleanup': '#4DAF6A',   # in-solution, crude + standard
}
BRANCH_LABELS = {
    'classic':   'original nanoPhos',
    'hybrid':    'SPEC + default enrichment',
    'wCleanup':  'In-solution digestion + cleanup + default enrichment',
    'woCleanup': 'In-solution digestion + default enrichment',
}
POINT_COLOR = '#393E46'      # Fig 2 convention for jittered points
GRID_COLOR  = '#F3F2F2'

## Data import & preprocessing

`REGENERATE = False` reads the committed result tables; `True` recomputes them from the
raw reports via `dp.regenerate_all` (reads ~2.7 GB once). Either path yields the same nine
tidy tables consumed by the figure panels below.

In [4]:
if REGENERATE:
    print(f'REGENERATE=True -> rebuilding data/ tables from raw reports in {RAW_DIR} ...')
    tables = dp.regenerate_all(RAW_DIR, RESULTS_DIR)
else:
    tables = dp.load_tables(RESULTS_DIR)

perrun    = tables['decoupling_perrun']              # 112 runs: depth / precursors / selectivity
summary   = tables['decoupling_summary']             # per (branch, input) means +/- SD
contrast  = tables['decoupling_contrasts']           # pairwise isolation contrasts (+ selectivity_diff_pp)
lin_sum   = tables['decoupling_linearity_summary']   # per-branch dilution-linearity summary
lin_sites = tables['decoupling_linearity_sites']     # per-site R^2 / slope (for histograms)
comp_cv   = tables['decoupling_completeness_cv']     # completeness + replicate CV
overlap   = tables['decoupling_overlap']             # Class I site-identity overlap vs classic
signal    = tables['decoupling_signal']              # phospho signal / dyn-range / loc / GRAVY
rt_profile= tables['decoupling_rt_profile']          # phospho signal vs retention time

print(f'branches: {sorted(perrun.branch.unique())}')
print(f'inputs:   {sorted(perrun.input_ng.unique())} ng')
print(f'runs:     {len(perrun)}  (expect 112 = 4 branches x 7 inputs x 4 reps)')
summary

branches: ['classic', 'hybrid', 'wCleanup', 'woCleanup']
inputs:   [np.int64(10), np.int64(20), np.int64(50), np.int64(100), np.int64(200), np.int64(500), np.int64(1000)] ng
runs:     112  (expect 112 = 4 branches x 7 inputs x 4 reps)


,branch,input_ng,n,classI_mean,classI_sd,prec_mean,prec_sd,selectivity_mean
0,classic,10,4.0,1666.8,91.3,5094.8,108.4,80.8
1,classic,20,4.0,3677.2,357.2,13148.2,396.2,92.1
2,classic,50,4.0,8581.8,628.5,30396.0,793.4,94.6
3,classic,100,4.0,12263.8,1094.8,42694.0,2117.7,88.7
4,classic,200,4.0,17924.8,230.9,63543.2,350.2,92.7
5,classic,500,4.0,23301.2,195.7,82002.8,393.2,83.6
6,classic,1000,4.0,25549.2,177.0,92239.0,598.6,85.8
7,hybrid,10,4.0,661.8,50.1,1675.8,135.0,17.3
8,hybrid,20,4.0,1838.0,50.1,4948.5,80.0,23.6
9,hybrid,50,4.0,4461.2,618.5,12902.2,1347.1,31.0


## Figures

Panels are built below from the tidy tables above (Fig-2 house style: `plotly_white`,
deep-red brand accent for `classic`, PDF export to `figures/decoupling/`).


In [5]:
# --- shared styling helpers (Fig 2 house style) -------------------------
# classic & hybrid share the SPEC preparation and differ ONLY in the
# phosphoenrichment, so they are the emphasized (solid, bold) pair that
# isolates the nanoPhos enrichment optimization. The in-solution branches
# (wCleanup, woCleanup) are drawn as lighter dashed lines for honest context.
EMPHASIZED = {'classic', 'hybrid'}

def _style(fig, ytitle, xtitle='Protein input (ng)', w=650, h=600, logx=True, yrange=None, legend=True):
    fig.update_layout(template='plotly_white', width=w, height=h, font=dict(size=12),
                      xaxis_title=xtitle, yaxis_title=ytitle,
                      legend_title='Workflow', showlegend=legend,
                      legend=dict(font=dict(size=10), yanchor='top', y=0.99, xanchor='left', x=0.01,
                                  bgcolor='rgba(255,255,255,0.6)'))
    if logx:
        fig.update_xaxes(type='log', tickvals=INPUT_ORDER, ticktext=[str(n) for n in INPUT_ORDER])
    fig.update_xaxes(gridcolor=GRID_COLOR)
    fig.update_yaxes(gridcolor=GRID_COLOR, rangemode='tozero' if yrange is None else 'normal',
                     range=yrange)
    return fig

def add_branch_series(fig, df, ycol, agg=True):
    """One series per branch vs input. classic/hybrid (shared SPEC prep) are
    emphasized as solid bold lines; wCleanup/woCleanup as lighter dashed lines.
    agg=True: per-rep points + mean+/-SD line (df = per-run table).
    agg=False: single mean line (df = per branch x input)."""
    for b in BRANCH_ORDER:
        sub = df[df.branch == b]
        if sub.empty:
            continue
        c = BRANCH_COLORS[b]
        emph = b in EMPHASIZED
        present = [n for n in INPUT_ORDER if n in set(sub['input_ng'])]
        if agg:
            fig.add_trace(go.Scatter(
                x=sub['input_ng'], y=sub[ycol], mode='markers', legendgroup=b, showlegend=False,
                marker=dict(size=7, color=c, opacity=0.5 if emph else 0.3,
                            line=dict(width=0.4, color='black'))))
            g = sub.groupby('input_ng')[ycol].agg(['mean', 'std']).reindex(present)
            err = dict(type='data', array=g['std'].fillna(0), visible=True, color=c, thickness=1, width=5)
        else:
            g = sub.set_index('input_ng').reindex(present).rename(columns={ycol: 'mean'})
            err = None
        fig.add_trace(go.Scatter(
            x=present, y=g['mean'], mode='lines+markers', name=BRANCH_LABELS[b], legendgroup=b,
            line=dict(color=c, width=(2.8 if b == 'classic' else 2.2) if emph else 1.5,
                      dash='solid' if emph else 'dash'),
            marker=dict(size=(10 if b == 'classic' else 8) if emph else 6, color=c,
                        line=dict(width=1, color='black')),
            opacity=1.0 if emph else 0.7,
            error_y=err))
    return fig

### Supplementary Figure 2a - Experimental design
Factor-decoupling of the nanoPhos workflow. An unstimulated HeLa lysate dilution series
(1 µg -> 10 ng, n = 4) is processed through four workflows sharing identical LC-MS/MS
(Evosep Eno + Orbitrap Astral Zoom). The two SPEC-based branches (**classic**, **hybrid**)
differ *only* in the phosphoenrichment - nanoPhos-optimized vs the default Agilent AssayMAP
Fe(III)-NTA protocol - so their difference isolates the phospho-specific optimizations
(modified SPEC elution, pre-elution buffer, loading/elution buffers, direct-to-Evotip
elution). The two lower branches replace SPEC with conventional in-solution preparation,
with (**wCleanup**) and without (**woCleanup**) SDB-RPS cleanup. Panel assembled externally
(schematic, panel *a*).

### Supplementary Figure 2b - Class I phosphosite depth vs input
Per-run unique Class I localized phosphosites (points = 4 replicates; line = mean ± SD).
All four workflows shown. The SPEC-based pair (**classic vs hybrid**, solid) isolates the
enrichment optimization: with the *same* SPEC preparation, the nanoPhos-optimized enrichment
roughly doubles depth across the whole range (~1.9-2.5×). The in-solution branches (dashed)
are shown for context - note `woCleanup` (crude lysate) competes on raw depth, which is why
selectivity (panel 2d) is the decisive axis.

In [6]:
fig = go.Figure()
add_branch_series(fig, perrun, 'classI_sites', agg=True)
_style(fig, 'Class I phosphosites')
fig.show()
fig.write_image('figures/decoupling/suppl_figure2b_depth_classI.pdf', width=600, height=600)

### Supplementary Figure 2c - Summed phosphopeptide-precursor intensity (classic vs hybrid, SPEC held constant)
Summed phosphopeptide-precursor intensity per run. With SPEC preparation identical, the
nanoPhos-optimized enrichment recovers **1.6-3.4× more phospho signal** than the default
Agilent enrichment across the range (fold annotated). Because only the enrichment differs,
this gain is attributable to the nanoPhos-specific chemistry. Only the two SPEC branches are
shown here on purpose: the uncleaned-lysate branch (`woCleanup`) yields even higher *raw*
signal by avoiding cleanup losses, but at poor selectivity - reported in the supplementary
quality table rather than headlined, since raw signal is not confident, selective coverage.

In [7]:
# R1d - phospho-specific ion signal, SPEC held constant (classic vs hybrid)
sig = signal.set_index(['branch', 'input_ng'])['phospho_sum_mean']
sd  = signal.set_index(['branch', 'input_ng'])['phospho_sum_sd']
x = INPUT_ORDER

fig = go.Figure()
for b in ('classic', 'hybrid'):
    y  = [sig.get((b, n), np.nan) for n in x]
    ey = [sd.get((b, n), 0)       for n in x]
    c = BRANCH_COLORS[b]
    fig.add_trace(go.Scatter(
        x=x, y=y, mode='lines+markers', name=BRANCH_LABELS[b], legendgroup=b,
        line=dict(color=c, width=2.8 if b == 'classic' else 2.2),
        marker=dict(size=10 if b == 'classic' else 8, color=c, line=dict(width=1, color='black')),
        error_y=dict(type='data', array=ey, visible=True, color=c, thickness=1, width=5)))

# fold-change (classic / hybrid) annotated above each classic point; SPEC prep identical
classic_y = [sig.get(('classic', n), np.nan) for n in x]
hybrid_y  = [sig.get(('hybrid',  n), np.nan) for n in x]
for n, cy, hy in zip(x, classic_y, hybrid_y):
    fig.add_annotation(x=np.log10(n), y=np.log10(cy), text=f'{cy / hy:.1f}×',
                       showarrow=False, yshift=16,
                       font=dict(size=10, color=BRANCH_COLORS['classic']))

_style(fig, 'Summed phosphopeptide precursor intensity', w=650, h=600, logx=True)
fig.update_yaxes(type='log', rangemode='normal')
fig.show()
fig.write_image('figures/decoupling/suppl_figure2c_phospho_signal.pdf', width=600, height=600)

### Supplementary Figure 2d - Phosphopeptide selectivity vs input
Phospho precursors / total precursors per run. **classic** is the only workflow that stays
> 80% across the whole range; the default-enrichment SPEC branch (**hybrid**) sits at
~20-35%, and the in-solution branches fail at opposite ends (`wCleanup` at low input,
`woCleanup` at high input). Read together with panel 2b: only the nanoPhos-optimized enrichment
delivers depth *and* selectivity simultaneously.

In [8]:
fig = go.Figure()
add_branch_series(fig, perrun, 'phospho_selectivity_pct', agg=True)
_style(fig, 'Phospho selectivity (%)', yrange=[0, 100], legend=False)
fig.add_hline(y=80, line=dict(color='grey', dash='dot', width=1))
fig.show()
fig.write_image('figures/decoupling/suppl_figure2d_selectivity.pdf', width=600, height=600)

### Supplementary Figure 2e - Replicate precision (median CV) vs input
Median coefficient of variation of Class I phosphosite intensities across the four replicates
(sites quantified in all four replicates; **lower = more reproducible**). nanoPhos (classic,
solid) is the most *consistent* workflow across the range: its CV declines smoothly from ~24%
at 10 ng to ~12% at 1 µg and it is never the worst at any input, whereas the SPEC +
standard-enrichment branch (hybrid) is erratic (up to ~39%). The crude woCleanup branch is
tight at low input but degrades at 1 µg (a noisy replicate). As elsewhere in this figure, the
message is robustness across the whole range rather than the single lowest value at every point.

In [9]:
# R1e - replicate precision (median CV) vs input, all four workflows
fig = go.Figure()
add_branch_series(fig, comp_cv, 'median_CV_pct', agg=False)
_style(fig, 'Median CV (%)', legend=False)
fig.show()
fig.write_image('figures/decoupling/suppl_figure2e_cv.pdf', width=600, height=600)

### Supplementary - quantification/identification quality
Reported for completeness (not a hero panel): replicate CV and data completeness (from the
Class I reports) and per-precursor localization confidence (from the precursor reports).
These are comparable across workflows or reflect classic's greater depth (more marginal
sites), so they are tabulated rather than plotted.

In [10]:
qual = comp_cv.merge(signal[['branch', 'input_ng', 'loc_median', 'loc_pct_ge90',
                             'phospho_sum_mean', 'gravy_pct_hydrophobic']],
                     on=['branch', 'input_ng'], how='left')
qual = qual[['branch', 'input_ng', 'mean_completeness_pct', 'pct_sites_in_all_reps',
             'median_CV_pct', 'loc_median', 'loc_pct_ge90',
             'phospho_sum_mean', 'gravy_pct_hydrophobic']]
qual = qual.sort_values(['branch', 'input_ng']).reset_index(drop=True)
qual.to_csv('data/decoupling_quality_supp.csv', index=False)
qual

,branch,input_ng,mean_completeness_pct,pct_sites_in_all_reps,median_CV_pct,loc_median,loc_pct_ge90,phospho_sum_mean,gravy_pct_hydrophobic
0,classic,10,62.8,33.1,24.4,0.333,37.5,1.000354e+06,10.0
1,classic,20,60.5,30.8,23.1,0.200,31.9,2.735658e+06,10.0
2,classic,50,63.5,34.7,22.0,0.167,34.0,1.047161e+07,10.8
3,classic,100,63.9,35.1,20.9,0.161,35.7,2.103836e+07,11.5
4,classic,200,67.9,42.2,13.6,0.128,38.1,6.342467e+07,11.5
5,classic,500,69.5,44.7,13.9,0.113,39.4,1.396064e+08,12.7
6,classic,1000,69.6,44.7,12.1,0.114,39.8,2.135908e+08,12.8
7,hybrid,10,65.9,37.2,38.8,0.447,41.6,4.003248e+05,7.0
8,hybrid,20,67.6,39.5,20.7,0.334,40.1,1.689603e+06,6.4
9,hybrid,50,61.4,32.2,37.8,0.249,39.2,5.320962e+06,7.3


## Metadata export

Per-panel source data → `MetaInfo_figures_checklist_v04.xlsx` (2a is the schematic, no data).

In [11]:
# === PRIDE MetaInfo export (run after all panels above) ===
import sys; sys.path.insert(0, r"src")
from metainfo_export import dump_panel
SFIG = 2   # figure number (single source of truth for sheet labels)
def _try(fn, sheet):
    try: fn()
    except Exception as e: print(f"  [SKIP {sheet}] {type(e).__name__}: {e}")

# 2a = experimental-design schematic (no data table)
# 2b - Class I phosphosite depth per run (all branches)
_try(lambda: dump_panel(perrun[["branch","input_ng","well","classI_sites"]], f"Suppl Figure {SFIG}b"), f"Suppl Figure {SFIG}b")
# 2c - summed phosphopeptide-precursor intensity (classic vs hybrid)
_try(lambda: dump_panel(signal[signal.branch.isin(["classic","hybrid"])][["branch","input_ng","phospho_sum_mean","phospho_sum_sd"]], f"Suppl Figure {SFIG}c"), f"Suppl Figure {SFIG}c")
# 2d - phosphopeptide selectivity per run (all branches)
_try(lambda: dump_panel(perrun[["branch","input_ng","well","phospho_selectivity_pct"]], f"Suppl Figure {SFIG}d"), f"Suppl Figure {SFIG}d")
# 2e - replicate median CV (all branches)
_try(lambda: dump_panel(comp_cv[["branch","input_ng","median_CV_pct"]], f"Suppl Figure {SFIG}e"), f"Suppl Figure {SFIG}e")
print("Suppl Figure 2 export done.")


  [MetaInfo] wrote 'Suppl Figure 2b'  (112 rows x 4 cols)
  [MetaInfo] wrote 'Suppl Figure 2c'  (14 rows x 4 cols)
  [MetaInfo] wrote 'Suppl Figure 2d'  (112 rows x 4 cols)
  [MetaInfo] wrote 'Suppl Figure 2e'  (28 rows x 3 cols)
Suppl Figure 2 export done.
